In [1]:
# cell 1

import os
from pathlib import Path

import pandas as pd

# -------------------------------------------------------------------
# PROJECT PATHS
# -------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "00_data" / "clean"  # where arcos_all.tsv lives
PROCESSED_DIR = PROJECT_ROOT / "00_data" / "processed"
DOCS_DIR = PROJECT_ROOT / "00_data" / "docs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Path to the huge ARCOS shipment file you download (see instructions doc)
ARCOS_FILE = RAW_DIR / "arcos_all.tsv"

# -------------------------------------------------------------------
# COLUMN MAP – UPDATE THESE IF YOUR RAW FILE USES DIFFERENT NAMES
# -------------------------------------------------------------------
COLS = {
    # geography
    "state": "BUYER_STATE",
    "county": "BUYER_COUNTY",  # or county FIPS if that exists
    "zip": "BUYER_ZIP",  # optional
    # timing
    "date": "TRANSACTION_DATE",  # often YYYYMMDD or date string
    # drug info
    "drug_name": "DRUG_NAME",
    "drug_code": "DRUG_CODE",  # optional, not used in our aggregation
    # shipment volume
    "quantity": "DOSAGE_UNIT",  # count of pills / units dispensed
    # MME-related columns (these vary a bit across ARCOS versions)
    # If your file has CALC_MME, change this to "CALC_MME"
    "mme": "MME",  # precomputed MME if present
    # Only needed if you must compute MME yourself:
    "strength_mg": "STRENGTH",  # mg per unit
    "mme_factor": "MME_CONVERSION_FACTOR",  # MME per mg
}

In [2]:
# cell 2

from typing import Optional


def load_data_dictionary(path: Path | str) -> pd.DataFrame:
    """Load the ARCOS data dictionary for reference (optional)."""
    path = Path(path)
    return pd.read_csv(path)


def compute_mme(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure we have an 'mme' column.

    Priority:
    1. If the raw file already has an MME column (COLS["mme"] or CALC_MME), rename it to 'mme'.
    2. Otherwise, try to compute mme = quantity * strength_mg * mme_factor.

    NOTE: You may need to tweak column names here based on your actual file.
    """
    # Case 1: precomputed MME column present
    mme_col_candidates = [COLS.get("mme"), "CALC_MME", "MME"]  # common names
    mme_col_candidates = [c for c in mme_col_candidates if c is not None]

    existing_mme_cols = [c for c in mme_col_candidates if c in df.columns]

    if existing_mme_cols:
        mme_col = existing_mme_cols[0]
        df = df.rename(columns={mme_col: "mme"})
        df["mme"] = pd.to_numeric(df["mme"], errors="coerce")
        return df

    # Case 2: compute from quantity * strength * factor
    qty_col = COLS.get("quantity")
    strength_col = COLS.get("strength_mg")
    factor_col = COLS.get("mme_factor")

    missing = [c for c in [qty_col, strength_col, factor_col] if c not in df.columns]
    if missing:
        raise ValueError(
            f"Cannot compute MME: missing columns {missing}. "
            "Either provide a precomputed MME column or update COLS mapping."
        )

    df["mme"] = (
        pd.to_numeric(df[qty_col], errors="coerce")
        * pd.to_numeric(df[strength_col], errors="coerce")
        * pd.to_numeric(df[factor_col], errors="coerce")
    )

    return df


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename key columns to simple names used downstream.

    We only rename if the raw columns are present to avoid KeyErrors.
    """
    rename_map = {}

    if COLS["state"] in df.columns:
        rename_map[COLS["state"]] = "state"
    if COLS["county"] in df.columns:
        rename_map[COLS["county"]] = "county"
    if COLS["date"] in df.columns:
        rename_map[COLS["date"]] = "date"
    if COLS["drug_name"] in df.columns:
        rename_map[COLS["drug_name"]] = "drug_name"
    if COLS["drug_code"] in df.columns:
        rename_map[COLS["drug_code"]] = "drug_code"
    if COLS["quantity"] in df.columns:
        rename_map[COLS["quantity"]] = "qty_pills"

    df = df.rename(columns=rename_map)
    return df

In [3]:
# cell 3

from typing import Iterable


def arcos_chunks(
    path: Path | str, chunksize: int = 1_000_000
) -> Iterable[pd.DataFrame]:
    """
    Generator that yields chunks of the ARCOS TSV file.

    We read with sep='\\t' because the FOIA file is tab-separated.
    """
    return pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize,
        low_memory=False,
    )

In [4]:
# cell 4

from tqdm.auto import tqdm


def process_to_county_year(path: Path) -> pd.DataFrame:
    """
    Reads the ARCOS file in chunks and returns a county-year dataset with:
      - total_pills
      - total_mme
      - n_transactions

    IMPORTANT:
    ----------
    We do *NOT* filter to specific molecule names (e.g. oxycodone/hydrocodone).
    Per professor guidance, the vast majority of ARCOS shipments in this period
    are prescription opioids, and we use total MME to capture potency-adjusted
    opioid supply.

    Aggregation level:
      (state, county, year)
    """

    aggregates: list[pd.DataFrame] = []

    for chunk in tqdm(arcos_chunks(path), desc="Processing ARCOS"):
        # --- 1. Standardize column names ---
        chunk = standardize_columns(chunk)

        # Sanity check: ensure required cols exist
        required = {"state", "county", "date", "qty_pills"}
        missing = required - set(chunk.columns)
        if missing:
            raise ValueError(f"Missing required columns in chunk: {missing}")

        # --- 2. Clean text fields ---
        chunk["state"] = chunk["state"].astype(str).str.upper().str.strip()
        chunk["county"] = chunk["county"].astype(str).str.upper().str.strip()

        if "drug_name" in chunk.columns:
            chunk["drug_name"] = chunk["drug_name"].astype(str).str.upper().str.strip()
        else:
            # If drug_name not present, create a dummy for counting transactions
            chunk["drug_name"] = "UNKNOWN"

        # --- 3. Parse date → year ---
        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
        chunk["year"] = chunk["date"].dt.year

        # Drop rows with missing key fields
        chunk = chunk.dropna(subset=["state", "county", "year"])

        # --- 4. Ensure numeric quantity + MME ---
        chunk["qty_pills"] = pd.to_numeric(chunk["qty_pills"], errors="coerce")

        # Compute or normalize MME
        chunk = compute_mme(chunk)
        chunk["mme"] = pd.to_numeric(chunk["mme"], errors="coerce")

        # Drop rows with no quantity or MME
        chunk = chunk.dropna(subset=["qty_pills", "mme"])

        if chunk.empty:
            continue

        # --- 5. Group to county-year for this chunk ---
        grouped = chunk.groupby(["state", "county", "year"], as_index=False).agg(
            total_pills=("qty_pills", "sum"),
            total_mme=("mme", "sum"),
            n_transactions=("drug_name", "size"),
        )
        aggregates.append(grouped)

    if not aggregates:
        raise ValueError("No data processed – check column mappings or file path.")

    # Combine across all chunks
    county_year = pd.concat(aggregates, ignore_index=True)

    # Ensure uniqueness across chunks (sum repeated keys)
    county_year = county_year.groupby(["state", "county", "year"], as_index=False).agg(
        total_pills=("total_pills", "sum"),
        total_mme=("total_mme", "sum"),
        n_transactions=("n_transactions", "sum"),
    )

    return county_year

In [5]:
# cell 5

county_year = process_to_county_year(ARCOS_FILE)

print("Final county-year shape:", county_year.shape)
display(county_year.head())

# Save in both CSV and Parquet
out_csv = PROCESSED_DIR / "arcos_county_year.csv"
out_parquet = PROCESSED_DIR / "arcos_county_year.parquet"

county_year.to_csv(out_csv, index=False)
county_year.to_parquet(out_parquet, index=False)

print("Saved:")
print(" -", out_csv)
print(" -", out_parquet)

Processing ARCOS: 0it [00:00, ?it/s]

Final county-year shape: (43822, 6)


,state,county,year,total_pills,total_mme,n_transactions
0,AA,NAN,2009,10.0,60.540,2
1,AA,NAN,2010,6.0,36.324,1
2,AE,NAN,2006,330.0,1362.150,2
3,AK,ALEUTIANS EAST,2006,300.0,908.100,1
4,AK,ALEUTIANS EAST,2007,100.0,302.700,1


Saved:
 - /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed/arcos_county_year.csv
 - /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed/arcos_county_year.parquet
